In [0]:
from utils.logger import get_logger
from etl_config.gold_dim_config import DIM_CONFIG, CATALOG, GOLD_SCHEMA, ColumnDef, SCD2_COLUMNS

logger = get_logger("gold_dim_setup")

In [0]:
def build_column_ddl(columns: list[ColumnDef]) -> str:
    lines = []
    for c in columns:
        nullability = "" if c.nullable else " not null"
        lines.append(f"{c.name} {c.sql_type}{nullability}")
    return ",\n        ".join(lines)

In [0]:
for dim_name, cfg in DIM_CONFIG.items():
    full_table_name = cfg.target_table

    columns = []

    if cfg.surrogate_key_name:
        columns.append(
            ColumnDef(
                cfg.surrogate_key_name,
                "bigint generated always as identity",
                nullable=False,
                comment="Surrogate key (PK, IDENTITY)",
            )
        )

    columns.extend(cfg.business_columns)

    if cfg.tracked_columns:
        columns.extend(SCD2_COLUMNS)

    col_ddl = ",\n        ".join(
        f"{c.name} {c.sql_type}" + ("" if c.nullable else " not null")
        for c in columns
    )

    spark.sql(f"""
        create table if not exists {full_table_name} (
            {col_ddl}
        )
        using delta
        comment '{cfg.table_description}'
    """)
    logger.info(f"Dimension table created/verified: {full_table_name}")

    spark.sql(f"""
        alter table {full_table_name}
        set tblproperties (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true',
            'description' = '{cfg.table_description}'
        )
    """)

    for c in columns:
        if c.comment:
            spark.sql(
                f"comment on column {full_table_name}.{c.name} is :comment",
                args={"comment": c.comment}
            )

    logger.info(f"Metadata applied for: {full_table_name}")

logger.info("Gold dimension setup complete.")